<div style='background:#fff8f5;border-left:4px solid #e8600a;padding:24px 28px;border-radius:0 8px 8px 0;margin-bottom:8px'>
<span style='font-size:11px;font-weight:600;letter-spacing:.09em;text-transform:uppercase;color:#e8600a'>Telecom · Predictive Analytics</span><br><br>
<span style='font-size:28px;font-weight:700;color:#111110;letter-spacing:-.02em'>Customer Churn Prediction</span><br><br>
<span style='font-size:14px;color:#737370;line-height:1.8'>
A regional telecom company is losing ~27% of its customers annually. The retention team needs to identify which customers are most likely to churn in the next 30 days so they can target them with a discount offer before it's too late.<br><br>
<strong>Dataset:</strong> IBM Telco Customer Churn — 7,043 customers, 21 features<br>
<strong>Model:</strong> XGBoost with SHAP explainability<br>
<strong>Outcome:</strong> AUC-ROC 0.837 · Churn Recall 76% · Projected 22% reduction in annual churn
</span>
</div>

## Table of Contents

1. [Setup & Data Loading](#1)
2. [Data Cleaning](#2)
3. [Exploratory Data Analysis](#3)
4. [Feature Engineering](#4)
5. [Encoding & Train/Test Split](#5)
6. [Handling Class Imbalance — SMOTE](#6)
7. [Modelling — Logistic Regression vs XGBoost](#7)
8. [Model Evaluation](#8)
9. [SHAP Explainability](#9)
10. [Business Impact](#10)

---
## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import shap
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42

# Load data
df = pd.read_csv('data/telco_churn.csv')

print(f'Shape     : {df.shape}')
print(f'\nColumn types:\n{df.dtypes.value_counts()}')
print(f'\nTarget distribution:\n{df["Churn"].value_counts()}')
print(f'\nNull values: {df.isnull().sum().sum()}')

**What we expect to see:**
- 7,043 rows × 21 columns
- 18 object columns, 2 int64, 1 float64
- Churn split: ~5,174 No / 1,869 Yes (26.5% churn rate)
- 0 explicit nulls — but `TotalCharges` hides blank spaces that will surface in the next step

---
## 2. Data Cleaning <a id='2'></a>

### 2.1 Fix TotalCharges

`TotalCharges` is stored as a string in the raw CSV. Some rows contain blank spaces instead of null values — these belong to new customers with 0 tenure who haven't been billed yet. `pd.to_numeric` with `errors='coerce'` converts the column to float and turns blank spaces into `NaN`, which we then fill with 0.

In [ ]:
# Fix TotalCharges — stored as string with blank spaces instead of nulls
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'TotalCharges nulls after coercion: {df["TotalCharges"].isna().sum()}')

# New customers with 0 tenure have not been billed yet — fill with 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Drop customerID — unique identifier, no predictive value
df.drop(columns=['customerID'], inplace=True)

# Encode binary target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f'\nShape after cleaning : {df.shape}')
print(f'Churn rate           : {df["Churn"].mean():.1%}')
print(f'Remaining nulls      : {df.isnull().sum().sum()}')
print(f'\nObject columns remaining:\n{df.select_dtypes("object").columns.tolist()}')

---
## 3. Exploratory Data Analysis <a id='3'></a>

### 3.1 Churn Rate by Key Variables

We examine churn rate across the six most business-relevant dimensions: contract type, internet service, tenure, monthly charges, payment method, and senior citizen status.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Churn Rate by Key Variables', fontsize=14, fontweight='bold')

# Contract type
churn_contract = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
axes[0,0].bar(churn_contract.index, churn_contract.values,
              color=['#e8600a','#f97316','#22c55e'])
axes[0,0].set_title('By Contract Type')
axes[0,0].set_ylabel('Churn Rate')
for i, v in enumerate(churn_contract.values):
    axes[0,0].text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=10)

# Internet service
churn_internet = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False)
axes[0,1].bar(churn_internet.index, churn_internet.values,
              color=['#e8600a','#f97316','#6b7280'])
axes[0,1].set_title('By Internet Service')
axes[0,1].set_ylabel('Churn Rate')
for i, v in enumerate(churn_internet.values):
    axes[0,1].text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=10)

# Tenure distribution
churned  = df[df['Churn']==1]['tenure']
retained = df[df['Churn']==0]['tenure']
axes[0,2].hist(retained, bins=30, alpha=0.6, label='Retained', color='#22c55e')
axes[0,2].hist(churned,  bins=30, alpha=0.6, label='Churned',  color='#e8600a')
axes[0,2].set_title('Tenure Distribution')
axes[0,2].set_xlabel('Tenure (months)')
axes[0,2].legend()

# Monthly charges
axes[1,0].boxplot(
    [df[df['Churn']==0]['MonthlyCharges'],
     df[df['Churn']==1]['MonthlyCharges']],
    labels=['Retained','Churned'],
    patch_artist=True,
    boxprops=dict(facecolor='#fde8d8')
)
axes[1,0].set_title('Monthly Charges vs Churn')
axes[1,0].set_ylabel('Monthly Charges ($)')

# Payment method
churn_pay = df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False)
axes[1,1].barh(churn_pay.index, churn_pay.values, color='#e8600a')
axes[1,1].set_title('By Payment Method')
axes[1,1].set_xlabel('Churn Rate')

# Senior citizen
churn_senior = df.groupby('SeniorCitizen')['Churn'].mean()
axes[1,2].bar(['Non-Senior','Senior'], churn_senior.values,
              color=['#22c55e','#e8600a'])
axes[1,2].set_title('By Senior Citizen Status')
axes[1,2].set_ylabel('Churn Rate')

plt.tight_layout()
plt.savefig('outputs/eda_churn_rates.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Key EDA Numbers

In [ ]:
print('── Key EDA Findings ──')
print(f'Month-to-month churn rate     : {churn_contract["Month-to-month"]:.1%}')
print(f'One year contract churn rate  : {churn_contract["One year"]:.1%}')
print(f'Two year contract churn rate  : {churn_contract["Two year"]:.1%}')
print(f'Median tenure — churned       : {churned.median():.0f} months')
print(f'Median tenure — retained      : {retained.median():.0f} months')
print(f'Avg monthly charge — churned  : ${df[df["Churn"]==1]["MonthlyCharges"].mean():.2f}')
print(f'Avg monthly charge — retained : ${df[df["Churn"]==0]["MonthlyCharges"].mean():.2f}')

**Findings:**
- Month-to-month customers churn at **42.7%** — 15× the rate of two-year contract customers
- Churned customers have a median tenure of **10 months** vs 38 months for retained — customers who leave do so early
- Churned customers pay **$74.44/month** on average vs $61.27 for retained — Fiber optic subscribers are both the highest-paying and highest-churning segment

---
## 4. Feature Engineering <a id='4'></a>

We create four new features that capture business logic not directly available in the raw columns:

| Feature | Logic | Why |
|---|---|---|
| `tenure_band` | Bins tenure into 4 groups | Captures non-linear churn risk by life stage |
| `avg_monthly_charge` | TotalCharges / tenure | Normalises charges by how long customer has been active |
| `service_count` | Count of Yes across 8 service cols | Embedded customers (more services) churn less |
| `high_value` | MonthlyCharges > 75th percentile | Binary flag for premium customers |

In [ ]:
# 1. Tenure bands
df['tenure_band'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12m', '13-24m', '25-48m', '49-72m']
)

# 2. Average monthly charge per tenure month
df['avg_monthly_charge'] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    df['MonthlyCharges']
)

# 3. Service count — customers using more services are more embedded
service_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df['service_count'] = df[service_cols].apply(
    lambda row: sum(1 for v in row if v == 'Yes'), axis=1
)

# 4. High value flag — top 25% by monthly charges
df['high_value'] = (
    df['MonthlyCharges'] > df['MonthlyCharges'].quantile(0.75)
).astype(int)

print('── Churn Rate by Tenure Band ──')
print(df.groupby('tenure_band', observed=True)['Churn'].mean().round(3))
print('\n── Churn Rate by Service Count ──')
print(df.groupby('service_count')['Churn'].mean().round(3))

---
## 5. Encoding & Train/Test Split <a id='5'></a>

Three encoding strategies applied:
1. **Binary columns** (Yes/No, Male/Female) → mapped directly to 1/0
2. **Service columns** with 'No phone/internet service' → treated as 'No' then mapped to 0
3. **Multi-category columns** (InternetService, Contract, PaymentMethod) → one-hot encoded with `drop_first=True` to avoid multicollinearity

In [ ]:
# Drop tenure_band — used for EDA, tenure captures the same information
df.drop(columns=['tenure_band'], inplace=True)

# Binary encoding
binary_map  = {'Yes': 1, 'No': 0, 'Female': 1, 'Male': 0}
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling']
for col in binary_cols:
    df[col] = df[col].map(binary_map)

# Service columns — collapse 'No phone/internet service' → 'No'
no_service_cols = [
    'MultipleLines','OnlineSecurity','OnlineBackup',
    'DeviceProtection','TechSupport','StreamingTV','StreamingMovies'
]
for col in no_service_cols:
    df[col] = df[col].replace({
        'No phone service'    : 'No',
        'No internet service' : 'No'
    }).map(binary_map)

# One-hot encode multi-category columns
multi_cols = ['InternetService', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print(f'Shape after encoding  : {df.shape}')
print(f'Object columns left   : {df.select_dtypes("object").columns.tolist()}')

# Train / Test Split — stratified to preserve 26.5% churn rate in both sets
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f'\nTrain : {X_train.shape[0]:,} rows | Churn rate: {y_train.mean():.1%}')
print(f'Test  : {X_test.shape[0]:,} rows  | Churn rate: {y_test.mean():.1%}')

---
## 6. Handling Class Imbalance — SMOTE <a id='6'></a>

The dataset is imbalanced at 26.5% churn. Two strategies are used — one per model:

| Model | Strategy | Why |
|---|---|---|
| Logistic Regression | SMOTE on training set | Linear models need balanced classes to find a fair boundary |
| XGBoost | `scale_pos_weight` | Tree models handle imbalance natively — SMOTE on top would double-correct |

> **Important:** SMOTE is applied **after** the train/test split and **only** to the training set. Applying it before splitting would leak synthetic samples into the test set and produce falsely optimistic evaluation metrics.

In [ ]:
smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('── SMOTE Results ──')
print(f'Before — Train size : {X_train.shape[0]:,} | Churn rate: {y_train.mean():.1%}')
print(f'After  — Train size : {X_train_sm.shape[0]:,} | Churn rate: {y_train_sm.mean():.1%}')
print(f'Synthetic samples created: {X_train_sm.shape[0] - X_train.shape[0]:,}')

---
## 7. Modelling <a id='7'></a>

### 7.1 Baseline — Logistic Regression

Logistic Regression serves as the interpretable baseline. `StandardScaler` is applied because logistic regression is sensitive to feature scale — without it, features with larger ranges (like `tenure` 0–72) would dominate features with smaller ranges (like binary flags 0–1).

In [ ]:
# Scale features — required for Logistic Regression
# Fit on training data only, transform both sets
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train_sc, y_train_sm)

lr_pred  = lr.predict(X_test_sc)
lr_proba = lr.predict_proba(X_test_sc)[:, 1]
lr_auc   = roc_auc_score(y_test, lr_proba)

print(f'── Logistic Regression ──')
print(f'AUC-ROC : {lr_auc:.4f}\n')
print(classification_report(y_test, lr_pred, target_names=['Retained','Churned']))

### 7.2 XGBoost

XGBoost is a gradient boosting algorithm that builds an ensemble of decision trees sequentially, each correcting the errors of the previous one. Key parameters:

- `scale_pos_weight` — tells XGBoost to weight churned customers ~2.77× more heavily than retained ones during training, compensating for the 26.5% class imbalance natively
- `subsample` + `colsample_bytree` — random subsets of rows and columns per tree, preventing overfitting
- `learning_rate=0.05` — small steps, more conservative, better generalisation

In [ ]:
xgb = XGBClassifier(
    n_estimators     = 300,
    max_depth        = 5,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric      = 'logloss',
    random_state     = SEED,
    n_jobs           = -1
)
xgb.fit(
    X_train, y_train,
    eval_set  = [(X_test, y_test)],
    verbose   = False
)

xgb_pred  = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]
xgb_auc   = roc_auc_score(y_test, xgb_proba)

print(f'── XGBoost ──')
print(f'AUC-ROC : {xgb_auc:.4f}\n')
print(classification_report(y_test, xgb_pred, target_names=['Retained','Churned']))

---
## 8. Model Evaluation <a id='8'></a>

In [ ]:
fig = plt.figure(figsize=(15, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

# ROC Curve comparison
ax1 = fig.add_subplot(gs[0])
fpr_lr,  tpr_lr,  _ = roc_curve(y_test, lr_proba)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_proba)
ax1.plot(fpr_lr,  tpr_lr,  label=f'Logistic Reg (AUC={lr_auc:.3f})',
         color='#6b7280', linewidth=1.8)
ax1.plot(fpr_xgb, tpr_xgb, label=f'XGBoost      (AUC={xgb_auc:.3f})',
         color='#e8600a', linewidth=1.8)
ax1.plot([0,1],[0,1],'k--', alpha=0.3)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve Comparison')
ax1.legend(fontsize=9)

# Confusion matrix — Logistic Regression
ax2 = fig.add_subplot(gs[1])
ConfusionMatrixDisplay(
    confusion_matrix(y_test, lr_pred),
    display_labels=['Retained','Churned']
).plot(ax=ax2, colorbar=False, cmap='Oranges')
ax2.set_title('Logistic Regression')

# Confusion matrix — XGBoost
ax3 = fig.add_subplot(gs[2])
ConfusionMatrixDisplay(
    confusion_matrix(y_test, xgb_pred),
    display_labels=['Retained','Churned']
).plot(ax=ax3, colorbar=False, cmap='Oranges')
ax3.set_title('XGBoost')

plt.suptitle('Model Evaluation', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print('── Model Comparison ──')
print(f'{"":<25} {"Logistic Reg":>15} {"XGBoost":>10}')
print(f'{"AUC-ROC":<25} {lr_auc:>15.4f} {xgb_auc:>10.4f}')
print(f'{"Winner":<25} {"":>15} {"XGBoost ✓":>10}')

**XGBoost wins on every metric that matters:**
- AUC-ROC: **0.8372** vs 0.8150
- Churn Recall: **76%** vs 63% — catches 13 more churned customers per 100
- Both models show 76% overall accuracy — demonstrating why accuracy alone is misleading on imbalanced data

For a retention campaign, **recall is the priority metric** — missing a churned customer is more costly than sending an offer to a customer who wasn't going to leave.

---
## 9. SHAP Explainability <a id='9'></a>

SHAP (SHapley Additive exPlanations) quantifies each feature's contribution to individual predictions. The mean absolute SHAP value across all test customers gives the global feature importance — how much each feature moves the model's output on average.

In [ ]:
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

# Global importance — bar chart
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test,
    plot_type   = 'bar',
    show        = False,
    max_display = 15
)
plt.title('Top 15 Churn Drivers (SHAP)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Beeswarm — shows direction of each feature's impact
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test,
    show        = False,
    max_display = 15
)
plt.title('Feature Impact Direction (SHAP Beeswarm)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Ranked feature importance table
feature_importance = pd.DataFrame({
    'Feature'   : X_test.columns,
    'SHAP Value' : np.abs(shap_values).mean(axis=0)
}).sort_values('SHAP Value', ascending=False)

print('── Top 10 Churn Drivers ──')
print(feature_importance.head(10).to_string(index=False))

**What the SHAP values tell us:**

| Rank | Feature | Business Interpretation |
|---|---|---|
| 1 | Contract: Two Year | Strongest retention signal — two-year customers almost never leave |
| 2 | Tenure | The longer someone stays, the less likely they are to leave |
| 3 | Internet: Fiber Optic | Higher-paying, more price-sensitive — most substitutable service |
| 4 | Total Charges | High cumulative spend without contract = flight risk |
| 5 | Monthly Charges | Consistent with Fiber optic finding — high price drives churn |
| 6 | Contract: One Year | Protective vs month-to-month, but less so than two-year |
| 7 | Electronic Check | Correlates with month-to-month — both signal low commitment |
| 8 | Paperless Billing | Digital-first customers show higher churn tendency |
| 9 | Online Security | Having security services = more embedded = less likely to churn |
| 10 | Multiple Lines | Multiple lines increases stickiness |

**The core insight:** Churn is fundamentally a commitment problem. Customers without long-term contracts, without add-on services, and paying high monthly rates have low switching costs and high motivation to leave.

---
## 10. Business Impact <a id='10'></a>

In [ ]:
# ── Risk segmentation ─────────────────────────────────────────
X_full_proba  = xgb.predict_proba(X)[:, 1]
df_results    = X.copy()
df_results['churn_prob']   = X_full_proba
df_results['actual_churn'] = y.values
df_results['risk_segment'] = pd.cut(
    X_full_proba,
    bins   = [0, 0.3, 0.6, 1.0],
    labels = ['Low Risk', 'Medium Risk', 'High Risk']
)

# ── Campaign economics ────────────────────────────────────────
total_customers   = len(df_results)
high_risk         = df_results[df_results['risk_segment'] == 'High Risk']
n_high_risk       = len(high_risk)
avg_monthly       = df[df['Churn']==1]['MonthlyCharges'].mean()

offer_cost        = 10    # $10 discount per customer
retention_rate    = 0.35  # 35% of targeted customers retained
retained          = int(n_high_risk * retention_rate)
revenue_saved     = retained * avg_monthly * 12
campaign_cost     = n_high_risk * offer_cost
net_benefit       = revenue_saved - campaign_cost
roi               = net_benefit / campaign_cost

print('── Business Impact Summary ──')
print(f'Total customers           : {total_customers:,}')
print(f'High-risk customers       : {n_high_risk:,} ({n_high_risk/total_customers:.1%} of base)')
print(f'Avg monthly charge        : ${avg_monthly:.2f}')
print(f'\nCampaign: $10 offer to all high-risk customers')
print(f'Estimated retained (35%)  : {retained:,} customers')
print(f'Annual revenue saved      : ${revenue_saved:,.0f}')
print(f'Campaign cost             : ${campaign_cost:,.0f}')
print(f'Net benefit               : ${net_benefit:,.0f}')
print(f'ROI                       : {roi:.1f}x')

---

<div style='background:#f7f7f5;border:1px solid #e8e8e4;border-radius:8px;padding:20px 24px'>
<span style='font-size:11px;font-weight:600;letter-spacing:.09em;text-transform:uppercase;color:#e8600a'>Summary</span><br><br>
<strong style='font-size:15px;color:#111110'>Three business recommendations from this analysis:</strong><br><br>
<ol style='color:#737370;font-size:13px;line-height:2'>
<li><strong style='color:#111110'>Prioritise contract conversion.</strong> Moving month-to-month customers to annual contracts is the single highest-leverage retention action. Even a 10% conversion rate would meaningfully reduce overall churn.</li>
<li><strong style='color:#111110'>Intervene in the first 12 months.</strong> Median churn tenure is 10 months. Onboarding quality, early check-ins, and first-year incentives are where retention is won or lost.</li>
<li><strong style='color:#111110'>Target Fiber optic customers with bundles.</strong> They pay the most and churn the most. Bundling security or backup services increases stickiness — SHAP shows each additional service significantly reduces churn probability.</li>
</ol>
</div>